# Phase 3 — Submission 07: Master Multi-Backbone Weighted Consensus Ensemble & Hub Audit

**Task:** Rhetorical Role Prediction in Legal Documents (Corporate Law — `CL` & Information Technology — `IT`)  
**Task Type:** Sentence-Level Multiclass Classification (13 Official Classes)  
**Primary Evaluation Metric:** Macro-F1 Score  
**Backbones Integrated:**
  1. Model 4 Peak Capacity Boundary Decoder (154M) — Sub 01 LoRA PEFT Anchor
  2. Model 1 Deep LLaMA Causal Decoder (135M) — Sub 02 Compulsory True LORMA Full Context
  3. Model 1 Deep LLaMA Causal Decoder (135M) — Sub 03 Rank-Stabilized rsLoRA
  4. Model 2 Wide Mistral Long-Flow Decoder (136M) — Sub 04 AdaLoRA Dynamic Rank Allocation
  5. Model 2 Wide Mistral Long-Flow Decoder (136M) — Sub 05 IA³ Vector Rescaling
  6. Model 3 Statutory Precision Full MHA Decoder (110M) — Sub 06 Compulsory Knowledge Distillation Student
**Target Hub Destination:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `Phase_3/sub07_master_ensemble/`)  

---

### Notebook Objectives
1. **Comprehensive Remote Hub Audit:** Systematically verify artifact completeness across all Phase 3 subfolders on `sarweshbarnwal/CS781-Legal-Models`.
2. **Comparative Visual Analytics:** Aggregate and plot validation Macro-F1 scores, training trajectories, and parameter efficiencies across all 7 submissions.
3. **Master Weighted Consensus Ensemble:** Fuse test predictions from the 6 diverse models weighted by their validation Macro-F1 performance.
4. **Codabench Submission Packaging:** Verify `prediction.json` formatting and package inside `checkpoint_sub07/submission.zip` with zero directory clutter (`assert len(infolist) == 1`).
5. **Atomic Publication:** Upload all ensemble artifacts and metadata to Hugging Face Hub in a single atomic commit.


In [ ]:
# Cell 2: Imports, Environment Setup & Upfront Hugging Face Authentication
import os
import sys
import json
import math
import time
import zipfile
import getpass
import datetime
from collections import Counter
from typing import Dict, List, Any, Optional, Tuple, Set

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download, list_repo_files

MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TARGET_SUBFOLDER = "Phase_3/sub07_master_ensemble"

LABEL2ID: Dict[str, int] = {
    "Fact": 0,
    "Issue": 1,
    "ArgumentPetitioner": 2,
    "ArgumentRespondent": 3,
    "Statute": 4,
    "Dissent": 5,
    "PrecedentReliedUpon": 6,
    "PrecedentNotReliedUpon": 7,
    "PrecedentOverruled": 8,
    "RulingByLowerCourt": 9,
    "RatioOfTheDecision": 10,
    "RulingByPresentCourt": 11,
    "None": 12,
}
ID2LABEL: Dict[int, str] = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 13
VALID_LABELS: Set[str] = set(LABEL2ID.keys())

# Upfront Write-Permission Token Check
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF") or os.environ.get("HUGGINGFACE_TOKEN") or ""
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HF")
    except Exception:
        pass

if HF_TOKEN:
    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO, exist_ok=True, token=HF_TOKEN)
        print(f"Successfully verified write permissions on Hugging Face Hub: '{MODEL_REPO}'")
    except Exception as e:
        print(f"Warning: Upfront HF token verification encountered: {e}")
else:
    print("Notice: HF_TOKEN not found in environment. You can enter it before upload.")


In [ ]:
# Cell 3: Comprehensive Remote Hugging Face Hub Audit
SUBMISSION_REGISTRY = {
    "sub00": {
        "title": "Sub 00: Model 1 Deep LLaMA 0-Context LORMA Ablation Baseline",
        "subfolder": "Phase_3/sub00_model1_lorma_0context",
        "arch_desc": "L=16, d=768, H_q=12, H_kv=4, SwiGLU d_ff=2304",
        "expected": [
            "model1_lorma_weights.pt", "mlp_head.pt", "config.json",
            "tokenizer.json", "tokenizer_config.json", "val_metrics.json",
            "training_evaluation_report.json", "loss_plot.png",
            "prediction.json", "submission.zip"
        ]
    },
    "sub01": {
        "title": "Sub 01: Model 4 LoRA PEFT Anchor & KD Teacher",
        "subfolder": "Phase_3/sub01_model4_lora_anchor",
        "arch_desc": "L=14, d=896, H_q=14, H_kv=7, GQA 2:1, SwiGLU d_ff=2432",
        "expected": [
            "adapter_model.safetensors", "adapter_config.json", "mlp_head.pt",
            "teacher_train_logits.pt", "config.json", "tokenizer.json",
            "tokenizer_config.json", "val_metrics.json",
            "training_evaluation_report.json", "loss_plot.png",
            "prediction.json", "submission.zip"
        ]
    },
    "sub02": {
        "title": "Sub 02: Model 1 Compulsory True LORMA Full Packed Context",
        "subfolder": "Phase_3/sub02_model1_lorma_context",
        "arch_desc": "L=16, d=768, H_q=12, H_kv=4, SwiGLU d_ff=2304",
        "expected": [
            "model1_lorma_weights.pt", "mlp_head.pt", "config.json",
            "tokenizer.json", "tokenizer_config.json", "val_metrics.json",
            "training_evaluation_report.json", "loss_plot.png",
            "prediction.json", "submission.zip"
        ]
    },
    "sub03": {
        "title": "Sub 03: Model 1 Rank-Stabilized rsLoRA",
        "subfolder": "Phase_3/sub03_model1_rslora",
        "arch_desc": "L=16, d=768, H_q=12, H_kv=4, SwiGLU d_ff=2304",
        "expected": [
            "adapter_model.safetensors", "adapter_config.json", "mlp_head.pt",
            "config.json", "tokenizer.json", "tokenizer_config.json",
            "val_metrics.json", "training_evaluation_report.json",
            "loss_plot.png", "prediction.json", "submission.zip"
        ]
    },
    "sub04": {
        "title": "Sub 04: Model 2 Wide Mistral AdaLoRA Dynamic Rank",
        "subfolder": "Phase_3/sub04_model2_adalora",
        "arch_desc": "L=12, d=896, H_q=14, H_kv=7, GQA 2:1, SwiGLU d_ff=2432",
        "expected": [
            "adapter_model.safetensors", "adapter_config.json", "mlp_head.pt",
            "config.json", "tokenizer.json", "tokenizer_config.json",
            "val_metrics.json", "training_evaluation_report.json",
            "loss_plot.png", "prediction.json", "submission.zip"
        ]
    },
    "sub05": {
        "title": "Sub 05: Model 2 Wide Mistral IA3 Vector Rescaling",
        "subfolder": "Phase_3/sub05_model2_ia3",
        "arch_desc": "L=12, d=896, H_q=14, H_kv=7, GQA 2:1, SwiGLU d_ff=2432",
        "expected": [
            "adapter_model.safetensors", "adapter_config.json", "mlp_head.pt",
            "config.json", "tokenizer.json", "tokenizer_config.json",
            "val_metrics.json", "training_evaluation_report.json",
            "loss_plot.png", "prediction.json", "submission.zip"
        ]
    },
    "sub06": {
        "title": "Sub 06: Model 3 Compulsory Knowledge Distillation Student",
        "subfolder": "Phase_3/sub06_model3_knowledge_distillation",
        "arch_desc": "L=12, d=768, H=12, SwiGLU d_ff=2048",
        "expected": [
            "model.safetensors", "mlp_head.pt", "config.json",
            "tokenizer.json", "tokenizer_config.json", "val_metrics.json",
            "training_evaluation_report.json", "loss_plot.png",
            "prediction.json", "submission.zip"
        ]
    }
}

api = HfApi(token=HF_TOKEN or None)
print(f"Scanning remote repository '{MODEL_REPO}' on Hugging Face Hub...")
try:
    repo_files = list_repo_files(repo_id=MODEL_REPO, token=HF_TOKEN or None)
    print(f"Retrieved {len(repo_files)} total files from repository.")
except Exception as e:
    print(f"Failed to access remote repository: {e}")
    repo_files = []

audit_results = {}
print("\n" + "="*80)
print(f"{'Submission':<10} {'Status':<10} {'Found/Expected':<16} {'Architecture Description'}")
print("="*80)

for key, meta in SUBMISSION_REGISTRY.items():
    subfolder = meta["subfolder"]
    expected = meta["expected"]
    found = [f[len(subfolder)+1:] for f in repo_files if f.startswith(subfolder + "/")]
    
    missing = []
    for exp in expected:
        if exp == "adapter_model.safetensors":
            if "adapter_model.safetensors" not in found and "adapter_model.bin" not in found:
                missing.append(exp)
        elif exp not in found:
            missing.append(exp)
            
    status = "COMPLETE" if not missing else ("PARTIAL" if found else "EMPTY")
    audit_results[key] = {
        "status": status,
        "found": len(found),
        "expected": len(expected),
        "missing": missing
    }
    print(f"{key:<10} {status:<10} {len(found)}/{len(expected):<14} {meta['arch_desc']}")
    if missing:
        print(f"   -> Missing: {missing}")
print("="*80 + "\n")


In [ ]:
# Cell 4: Load Soft Probabilities & Validation Metrics across Sub 01 through Sub 06
sub_keys = ["sub01", "sub02", "sub03", "sub04", "sub05", "sub06"]
all_probs: Dict[str, Dict[str, torch.Tensor]] = {}
all_predictions: Dict[str, Dict[str, List[str]]] = {}
val_f1_scores: Dict[str, Optional[float]] = {}

for key in sub_keys:
    subfolder = SUBMISSION_REGISTRY[key]["subfolder"]
    local_dir = f"checkpoint_{key}"
    
    # 1. Validation F1 Score Lookup (None if absent - no invented defaults)
    dev_f1: Optional[float] = None
    local_val = f"{local_dir}/val_metrics.json"
    if os.path.exists(local_val):
        try:
            with open(local_val, "r", encoding="utf-8") as f:
                data = json.load(f)
                val = data.get("best_dev_macro_f1", data.get("dev_macro_f1"))
                if val is not None:
                    dev_f1 = float(val)
        except Exception:
            pass
    else:
        try:
            hub_val = hf_hub_download(MODEL_REPO, f"{subfolder}/val_metrics.json", token=HF_TOKEN or None)
            with open(hub_val, "r", encoding="utf-8") as f:
                data = json.load(f)
                val = data.get("best_dev_macro_f1", data.get("dev_macro_f1"))
                if val is not None:
                    dev_f1 = float(val)
        except Exception:
            pass
    val_f1_scores[key] = dev_f1

    # 2. Soft Logits / Probabilities Lookup
    test_loaded = False
    local_logits = f"{local_dir}/test_logits.pt"
    if os.path.exists(local_logits):
        try:
            obj = torch.load(local_logits, map_location="cpu")
            probs = {d_id: F.softmax(t if isinstance(t, torch.Tensor) else torch.tensor(t, dtype=torch.float32), dim=-1) for d_id, t in obj.items()}
            all_probs[key] = probs
            test_loaded = True
            print(f"Loaded soft test_logits locally for {key}: {len(probs)} documents | Dev F1: {dev_f1}")
        except Exception as e:
            print(f"Warning: Failed loading local test_logits for {key}: {e}")

    if not test_loaded:
        try:
            hub_logits = hf_hub_download(MODEL_REPO, f"{subfolder}/test_logits.pt", token=HF_TOKEN or None)
            obj = torch.load(hub_logits, map_location="cpu")
            probs = {d_id: F.softmax(t if isinstance(t, torch.Tensor) else torch.tensor(t, dtype=torch.float32), dim=-1) for d_id, t in obj.items()}
            all_probs[key] = probs
            test_loaded = True
            print(f"Downloaded soft test_logits from Hub for {key}: {len(probs)} documents | Dev F1: {dev_f1}")
        except Exception:
            pass

    # 3. Fallback to discrete prediction.json
    local_pred = f"{local_dir}/prediction.json"
    if os.path.exists(local_pred):
        with open(local_pred, "r", encoding="utf-8") as f:
            all_predictions[key] = json.load(f)
    else:
        try:
            hub_pred = hf_hub_download(MODEL_REPO, f"{subfolder}/prediction.json", token=HF_TOKEN or None)
            with open(hub_pred, "r", encoding="utf-8") as f:
                all_predictions[key] = json.load(f)
        except Exception:
            pass

print(f"\nGathered soft probabilities from {len(all_probs)} models and discrete predictions from {len(all_predictions)} models.")


In [ ]:
# Cell 5: Master Weighted Consensus Ensemble Engine & Strict Codabench Packaging
os.makedirs("checkpoint_sub07", exist_ok=True)

# Select available models (prefer soft probability models)
available_models = list(all_probs.keys()) if all_probs else list(all_predictions.keys())

if not available_models:
    print("Notice: No prediction files found locally or on Hub yet.")
    print("To generate predictions, run Sub 01 through Sub 06 training notebooks.")
else:
    # 1. Determine model weights based on validation Macro-F1
    valid_scores = {m: val_f1_scores[m] for m in available_models if val_f1_scores.get(m) is not None and val_f1_scores[m] > 0}
    if len(valid_scores) == len(available_models):
        print("Using validation Macro-F1 score weighting across available models:")
        total_w = sum(valid_scores.values())
        weights = {m: w / total_w for m, w in valid_scores.items()}
    else:
        print("Notice: Validation F1 scores not fully available; using uniform weighting:")
        weights = {m: 1.0 / len(available_models) for m in available_models}

    for m, w in weights.items():
        print(f"  * {m}: weight = {w:.4f} (Dev F1 = {val_f1_scores.get(m)})")

    ensemble_predictions: Dict[str, List[str]] = {}

    if all_probs:
        # Soft Probability Averaging Consensus
        first_m = available_models[0]
        test_doc_ids = list(all_probs[first_m].keys())
        for doc_id in test_doc_ids:
            N_sents = all_probs[first_m][doc_id].shape[0]
            ens_probs = torch.zeros((N_sents, NUM_CLASSES), dtype=torch.float32)
            for m in available_models:
                if doc_id in all_probs[m]:
                    ens_probs += weights[m] * all_probs[m][doc_id]
            preds = ens_probs.argmax(dim=-1).tolist()
            ensemble_predictions[doc_id] = [ID2LABEL[idx] for idx in preds]
    else:
        # Discrete Voting Consensus Fallback
        first_m = available_models[0]
        test_doc_ids = list(all_predictions[first_m].keys())
        for doc_id in test_doc_ids:
            sent_count = len(all_predictions[first_m][doc_id])
            doc_ensemble = []
            for s_idx in range(sent_count):
                label_votes = Counter()
                for m in available_models:
                    if doc_id in all_predictions[m] and s_idx < len(all_predictions[m][doc_id]):
                        lbl = all_predictions[m][doc_id][s_idx]
                        label_votes[lbl] += weights[m]
                doc_ensemble.append(label_votes.most_common(1)[0][0])
            ensemble_predictions[doc_id] = doc_ensemble

    # Save ensemble prediction.json
    pred_path = "checkpoint_sub07/prediction.json"
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(ensemble_predictions, f, indent=2)

    total_sents = sum(len(v) for v in ensemble_predictions.values())
    print(f"\nConstructed ensemble predictions for {len(ensemble_predictions)} documents ({total_sents:,} sentences).")

    # Package strictly into submission.zip with prediction.json at root
    zip_path = "checkpoint_sub07/submission.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")

    # Strict Codabench Integrity Verification
    with zipfile.ZipFile(zip_path, "r") as zf:
        infolist = zf.infolist()
        assert len(infolist) == 1, f"Codabench Error: Expected exactly 1 file in submission.zip, found {len(infolist)}"
        assert infolist[0].filename == "prediction.json", f"Codabench Error: Found {infolist[0].filename} instead of prediction.json"

    # Verify label domain and no None values
    for d_id, preds in ensemble_predictions.items():
        assert None not in preds, f"None found in doc {d_id}"
        for lbl in preds:
            assert lbl in VALID_LABELS, f"Invalid label '{lbl}' in doc {d_id}"

    print(f"Verified submission.zip root integrity: {os.path.getsize(zip_path):,} bytes, root file: 'prediction.json'.")

    # Save metadata with dynamic UTC timestamp
    ensemble_meta = {
        "ensemble_strategy": "Optimal Validation Macro-F1 Weighted Soft Probability Fusion" if all_probs else "F1-Weighted Consensus Voting",
        "models": available_models,
        "weights": weights,
        "val_f1_scores": val_f1_scores,
        "total_docs": len(ensemble_predictions),
        "total_sentences": total_sents,
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat()
    }
    with open("checkpoint_sub07/ensemble_metadata.json", "w", encoding="utf-8") as f:
        json.dump(ensemble_meta, f, indent=2)
    with open("checkpoint_sub07/val_comparison.json", "w", encoding="utf-8") as f:
        json.dump(val_f1_scores, f, indent=2)
    print("Saved ensemble_metadata.json and val_comparison.json.")


In [ ]:
# Cell 6: Comparative Performance Visualization across All Models
if val_f1_scores:
    model_names = list(val_f1_scores.keys())
    scores = [val_f1_scores[k] for k in model_names]
    
    plt.figure(figsize=(10, 5))
    bars = plt.bar(model_names, scores, color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"][:len(model_names)])
    plt.ylim([0.0, 1.0])
    plt.xlabel("Phase 3 Model Submissions")
    plt.ylabel("Dev Macro-F1 Score")
    plt.title("Phase 3: Validation Macro-F1 Score Comparison")
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.4f}", ha="center", va="bottom", fontweight="bold")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig("checkpoint_sub07/comparison_plot.png", dpi=150)
    plt.show()
    print("Saved comparison visualization to 'checkpoint_sub07/comparison_plot.png'.")


In [ ]:
# Cell 7: Atomic Single-Commit Upload of Sub 07 to Hugging Face Hub
output_dir = "checkpoint_sub07"

if os.path.exists(os.path.join(output_dir, "submission.zip")):
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass("Enter Hugging Face Write Token: ")
        
    readme_path = os.path.join(output_dir, "README.md")
    readme_content = f"""---
base_model: {MODEL_REPO}
tags:
- legal
- legal-nlp
- rhetorical-roles
---

# Phase 3 Sub 07: Master Multi-Model Weighted Probability Ensemble
- **Target Repository:** `{MODEL_REPO}`
- **Task:** Sentence-Level Rhetorical Role Prediction (13 Classes)
"""
    with open(readme_path, "w", encoding="utf-8") as rf:
        rf.write(readme_content)

    api = HfApi(token=HF_TOKEN)
    print(f"Initiating atomic upload of Sub 07 ensemble to '{MODEL_REPO}' under '{TARGET_SUBFOLDER}'...")
    api.upload_folder(
        folder_path=output_dir,
        repo_id=MODEL_REPO,
        path_in_repo=TARGET_SUBFOLDER,
        commit_message="Upload Phase 3 Sub 07 Master Multi-Backbone Weighted Consensus Ensemble",
        token=HF_TOKEN
    )
    print(f"\nUpload complete! Artifacts published to:\nhttps://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")
else:
    print("Ensemble zip not yet created. Complete the ensemble step first.")
